## 00 — Start here

Purpose: a copy/paste workbench for unfamiliar tabular prediction problems. Run the core cells in order; optional recipes are disabled until selected. The default is a small synthetic regression demo, never a claim about real financial data.

Fast path: 01–06 → 08–11 → 13–15 → 18. Change configuration in 01, replace the demo in 02, and follow the same numbered headings in this guide and notebook. Each code block states its dependencies through the variables it uses. Keep notebook outputs clear when sharing real data.

Six-hour exercise: 0–30 min framing/integrity; 30–75 train-only EDA and split design; 75–150 baselines; 150–240 one nonlinear model and diagnostics; 240–300 robustness/interpretation; final hour results/presentation. Adapt to your actual deadline.

Before coding ask: what does a row represent, what is predicted, at what time, using what available information, and what decision does prediction support? Write the answers down. Never infer availability from a column name alone.

Dependencies: Python 3.8+; numpy, pandas, matplotlib, scipy, scikit-learn. Optional inference recipes use statsmodels. Install in your own environment with `python -m pip install numpy pandas matplotlib scipy scikit-learn statsmodels jupyter`. Check package versions on the supplied machine; the encoder recipe supports old and new sklearn APIs.

## 01 — Imports and configuration

Set TASK to regression or classification; SPLIT to iid, time, or group. TIME identifies prediction timestamps; GROUP identifies entities that must stay apart for unseen-entity evaluation. For binary classification POS_LABEL is the event you care about. Choose features explicitly after inspecting the schema. Set GAP_TIME_BLOCKS from label overlap/availability, not by tuning performance.

In [ ]:
import copy, inspect, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import KFold, StratifiedKFold, GroupKFold, GroupShuffleSplit, TimeSeriesSplit, train_test_split, GridSearchCV
from sklearn.dummy import DummyRegressor, DummyClassifier
from sklearn.linear_model import LinearRegression, Ridge, Lasso, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier, GradientBoostingRegressor, GradientBoostingClassifier
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, accuracy_score, balanced_accuracy_score, log_loss, roc_auc_score, average_precision_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.inspection import permutation_importance, PartialDependenceDisplay
from IPython.display import display
SEED = 42
TASK = 'regression'              # or 'classification'
SPLIT = 'iid'                    # or 'time', 'group'
TARGET = 'target'
TIME = None                      # e.g. 'timestamp'
GROUP = None                     # e.g. 'entity'
POS_LABEL = 1
FEATURES = None                  # preferably an explicit allowlist
DROP = []                        # IDs, future information, target proxies
GAP_TIME_BLOCKS = 0
N_SPLITS = 3
plt.rcParams.update({'figure.figsize': (8, 4), 'axes.grid': True, 'font.size': 10})

## 02 — Load data or run the demo

Replace only the demo block with pd.read_csv/path, read_parquet, read_excel, or your existing DataFrame. Preserve a raw copy. Parse dates explicitly and verify timezone/units. Join external tables using validate='many_to_one' and inspect unmatched rows; a many-to-many join can silently inflate your dataset.

In [ ]:
# Replace this block with: df = pd.read_csv('data.csv')
rng = np.random.RandomState(SEED)
n = 500
df = pd.DataFrame({'x1': rng.normal(size=n), 'x2': rng.normal(size=n),
                   'sector': rng.choice(['A', 'B', 'C'], n),
                   'timestamp': pd.date_range('2020-01-01', periods=n, freq='h'),
                   'entity': np.arange(n) % 25})
signal = 2*df.x1 - df.x2 + 0.8*(df.sector == 'B')
df[TARGET] = signal + rng.normal(size=n)
if TASK == 'classification':
    df[TARGET] = (df[TARGET] > 0).astype(int)
df.loc[rng.choice(n, 30, replace=False), 'x2'] = np.nan
if FEATURES is None:
    FEATURES = ['x1', 'x2', 'sector']  # change for your real dataset
if SPLIT == 'time' and TIME is None:
    TIME = 'timestamp'
if SPLIT == 'group' and GROUP is None:
    GROUP = 'entity'
raw = df.copy(deep=True)
display(df.head(), pd.Series({'rows': len(df), 'columns': df.shape[1]}))

## 03 — Integrity and schema

Question → code: corrupt values? dtype/range/infinity checks. Missingness? missing fractions and patterns. Duplicate observations? duplicated plus a domain key. Repeated IDs? nunique and group sizes. Do not delete repeated rows automatically: repetitions may be legitimate. Structural checks can cover the input, but keep target relationship exploration in development data. Missing targets are excluded from supervised fitting; never impute them.

In [ ]:
assert df.columns.is_unique, 'Duplicate column names'
assert TARGET in df
schema = pd.DataFrame({'dtype': df.dtypes.astype(str), 'missing': df.isna().mean(),
                       'unique': df.nunique(dropna=False)})
display(schema, df.describe(include='all').T.head(30))
print('Exact duplicate rows:', df.duplicated().sum())
numeric = df.select_dtypes(include=np.number)
display(pd.Series(np.isinf(numeric).sum(), name='infinities'))
# Domain-specific: assert df['price'].dropna().gt(0).all()
# Key-specific: display(df[df.duplicated(['entity', 'timestamp'], keep=False)])
df = df.loc[df[TARGET].notna()].copy()
df = df.reset_index(drop=True)
if TIME:
    df[TIME] = pd.to_datetime(df[TIME], errors='raise')
    assert df[TIME].notna().all()
    df = df.sort_values(TIME, kind='mergesort').reset_index(drop=True)
if GROUP:
    assert df[GROUP].notna().all(), 'Resolve missing group IDs explicitly'
FEATURES = [c for c in FEATURES if c not in set(DROP + [TARGET, TIME, GROUP])]
assert FEATURES, 'Choose at least one feature'
assert set(FEATURES).issubset(df.columns)

## 04 — Reserve the holdout and design folds

IID: shuffled folds only if rows are exchangeable. Time: split distinct timestamps so simultaneous rows never straddle a boundary. The gap below counts timestamp blocks, not rows; irregular times/overlapping horizon labels require purging by actual label-end timestamps. Group: hold out whole entities for unseen-entity deployment. Forecasting known entities requires temporal validation instead. If both time and entity separation are required, implement a custom splitter matching deployment. Never use a random split to claim forecasting performance.

The final holdout is for one frozen evaluation. Use development folds to choose models/features. EDA on development data can still overfit model selection; keep experiments few and record them.

In [ ]:
if SPLIT == 'time':
    assert TIME
    times = np.sort(df[TIME].unique())
    cut = int(0.8 * len(times))
    assert cut > GAP_TIME_BLOCKS
    dev = df[df[TIME].isin(times[:cut-GAP_TIME_BLOCKS])].copy()
    holdout = df[df[TIME].isin(times[cut:])].copy()
elif SPLIT == 'group':
    assert GROUP
    a, b = next(GroupShuffleSplit(n_splits=1, test_size=.2, random_state=SEED).split(df, groups=df[GROUP]))
    dev, holdout = df.iloc[a].copy(), df.iloc[b].copy()
else:
    dev, holdout = train_test_split(df, test_size=.2, random_state=SEED,
        stratify=df[TARGET] if TASK == 'classification' else None)
dev, holdout = dev.reset_index(drop=True), holdout.reset_index(drop=True)
X, y = dev[FEATURES].copy(), dev[TARGET].copy()
if SPLIT == 'time':
    times = np.sort(dev[TIME].unique())
    # Manual gap supports older sklearn versions too.
    cv = []
    for a, b in TimeSeriesSplit(n_splits=N_SPLITS).split(times):
        if GAP_TIME_BLOCKS:
            a = a[:-GAP_TIME_BLOCKS]
        assert len(a), 'Gap leaves no training timestamps'
        cv.append((np.flatnonzero(dev[TIME].isin(times[a])), np.flatnonzero(dev[TIME].isin(times[b]))))
elif SPLIT == 'group':
    cv = list(GroupKFold(n_splits=N_SPLITS).split(X, y, groups=dev[GROUP]))
else:
    splitter = StratifiedKFold(N_SPLITS, shuffle=True, random_state=SEED) if TASK == 'classification' else KFold(N_SPLITS, shuffle=True, random_state=SEED)
    cv = list(splitter.split(X, y))
display(pd.DataFrame([{'fold': i, 'train': len(a), 'validation': len(b)} for i,(a,b) in enumerate(cv)]))
print('Development:', len(dev), '| untouched holdout:', len(holdout))

## 05 — Target and missingness EDA

Does target shape change the loss? Regression: inspect tails, quantiles and skew; MSE emphasizes large errors, MAE the conditional median. Classification: inspect imbalance and practical error costs. Missingness can be informative but can also represent a changing collection process. Add missing indicators inside preprocessing. Investigate outliers before winsorizing; fit any thresholds on each training fold. Avoid dropping difficult validation observations.

In [ ]:
display(y.describe())
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
if TASK == 'regression':
    y.hist(bins=35, ax=axes[0]); axes[0].set_title('Development target distribution')
    y.plot.box(ax=axes[1]); axes[1].set_title('Development target tails')
else:
    y.value_counts().plot.bar(ax=axes[0]); axes[0].set_title('Development class counts')
    y.value_counts(normalize=True).plot.bar(ax=axes[1]); axes[1].set_title('Development class proportions')
plt.tight_layout(); plt.show()
display(X.isna().mean().sort_values(ascending=False).head(20))
display(X.isna().groupby(y).mean().head(15))

## 06 — Relationships, redundancy and categories

Linear signal? Pearson correlation. Monotone signal? Spearman. Nonlinear structure? scatter/binned target means, optionally mutual information (training only; noisy, not a significance test). Redundant predictors? correlation/conditioning; Ridge often helps. Categories? show counts alongside target averages: tiny groups produce misleading extremes. Zero correlation does not imply independence; correlated features make importance attribution ambiguous.

In [ ]:
num = X.select_dtypes(include=np.number).columns.tolist()
cat = [c for c in X if c not in num]
if TASK == 'regression' and num:
    display(pd.DataFrame({'pearson': X[num].corrwith(y), 'spearman': X[num].corrwith(y, method='spearman')}).sort_values('spearman', key=lambda s:s.abs(), ascending=False))
if num:
    corr = X[num[:15]].corr()
    fig, ax = plt.subplots(figsize=(7, 6))
    im = ax.imshow(corr, vmin=-1, vmax=1, cmap='coolwarm')
    ax.set_xticks(range(len(corr))); ax.set_xticklabels(corr.columns, rotation=90)
    ax.set_yticks(range(len(corr))); ax.set_yticklabels(corr.columns)
    ax.set_title('Development feature correlation'); fig.colorbar(im, ax=ax)
    plt.tight_layout(); plt.show()
if TASK == 'regression':
    for c in num[:3]:
        plt.scatter(X[c], y, s=10, alpha=.3); plt.xlabel(c); plt.ylabel(TARGET)
        plt.title('Development relationship'); plt.show()
        bins = pd.qcut(X[c], 8, duplicates='drop')
        display(pd.DataFrame({'bin': bins, 'y': y}).groupby('bin', observed=True).y.agg(['count','mean','std']))
for c in cat[:3]:
    display(dev.groupby(c, dropna=False)[TARGET].agg(['count', 'mean']) if TASK == 'regression' else pd.crosstab(dev[c], y, normalize='index'))

## 07 — Optional time and feature recipes

Set RUN_TIME_EDA / RUN_FEATURE_RECIPE to True only when appropriate. Temporal trends? target rolling mean/std and feature-target correlations per time bucket. Lags and rolling means must respect entity boundaries, chronological order and actual availability. A shifted target is allowed only when that past label has matured at prediction time. Duplicate entity timestamps need an explicit aggregation/order rule. Log1p is only defined for x > -1 and is not automatically suitable for signed returns. Do not run cointegration on returns when the hypothesis concerns stationary combinations of nonstationary price levels: check integration order first.

In [ ]:
RUN_TIME_EDA = False
RUN_FEATURE_RECIPE = False
if RUN_TIME_EDA:
    assert TIME and TASK == 'regression'
    z = dev.groupby(TIME)[TARGET].mean().sort_index()
    z.plot(alpha=.4, label='target')
    z.rolling(20, min_periods=10).mean().plot(label='rolling mean')
    plt.legend(); plt.title('Development target through time'); plt.show()
    z.rolling(20, min_periods=10).std().plot(title='Rolling target standard deviation'); plt.show()
if RUN_FEATURE_RECIPE:
    assert TIME
    c = num[0]  # choose a measured feature, not an unavailable label
    ordered = df.sort_values(([GROUP] if GROUP else []) + [TIME]).copy()
    if GROUP:
        assert not ordered.duplicated([GROUP, TIME]).any()
        ordered[c+'_lag1'] = ordered.groupby(GROUP)[c].shift(1)
        ordered[c+'_past_mean'] = ordered.groupby(GROUP)[c].transform(lambda s: s.shift(1).rolling(10, min_periods=3).mean())
    else:
        assert ordered[TIME].is_unique
        ordered[c+'_lag1'] = ordered[c].shift(1)
        ordered[c+'_past_mean'] = ordered[c].shift(1).rolling(10, min_periods=3).mean()
    # Rebuild splits and FEATURES after adopting ordered as df.
    display(ordered.head())

## 08 — Fold-safe preprocessing

Everything learned from values belongs in a pipeline: imputation, scaling, encoding, PCA, feature selection and clipping thresholds. Numeric-coded categories must be recast explicitly. IDs are not automatically predictors. Unseen categories use handle_unknown='ignore'; high-cardinality one-hot features can exhaust memory. This intentionally dense template fits small interview data; use sparse preprocessing plus compatible models for large data. All-missing numeric columns should be investigated and deliberately excluded if unusable.

In [ ]:
num = X.select_dtypes(include=np.number).columns.tolist()
cat = [c for c in X if c not in num]
# Normalize categorical missing values; no statistics are learned here.
for c in cat:
    X[c] = X[c].astype(object).where(X[c].notna(), np.nan)
encoder_args = {'handle_unknown': 'ignore'}
encoder_args['sparse_output' if 'sparse_output' in inspect.signature(OneHotEncoder).parameters else 'sparse'] = False
preprocess = ColumnTransformer([
    ('numeric', Pipeline([('impute', SimpleImputer(strategy='median', add_indicator=True)), ('scale', StandardScaler())]), num),
    ('category', Pipeline([('impute', SimpleImputer(strategy='constant', fill_value='__MISSING__')), ('encode', OneHotEncoder(**encoder_args))]), cat)
], remainder='drop')
def make_pipeline(model):
    return Pipeline([('preprocess', clone(preprocess)), ('model', model)])

## 09 — Baselines and candidate models

Begin with mean/median or class-prior baseline, then a linear/regularized model and one nonlinear competitor. Ridge handles correlated predictors with shrinkage; Lasso encourages sparse coefficients but can select unstably among correlated features. Logistic regression provides class probabilities; evaluate calibration too. Forests capture nonlinear interactions but may overfit small samples and do not extrapolate well. PCA preserves feature variance, not target signal; validate it rather than assuming it helps. Avoid broad hyperparameter searches on a short exercise.

In [ ]:
if TASK == 'regression':
    models = {'baseline': make_pipeline(DummyRegressor(strategy='mean')),
              'ols': make_pipeline(LinearRegression()),
              'ridge': make_pipeline(Ridge(alpha=10.0)),
              'forest': make_pipeline(RandomForestRegressor(n_estimators=100, min_samples_leaf=8, max_features=.8, random_state=SEED, n_jobs=1))}
    PRIMARY = 'mse'; SCORING = 'neg_mean_squared_error'
else:
    models = {'baseline': make_pipeline(DummyClassifier(strategy='prior')),
              'logistic': make_pipeline(LogisticRegression(C=1.0, max_iter=1000)),
              'forest': make_pipeline(RandomForestClassifier(n_estimators=100, min_samples_leaf=8, random_state=SEED, n_jobs=1))}
    PRIMARY = 'logloss'; SCORING = 'neg_log_loss'

## 10 — Compare out-of-fold predictions

Record fold metrics and variability, not only the best score. Classification probabilities must be aligned by class label. A temporal/group fold missing classes can make a classifier or AUC invalid: redesign/report it; never silently discard the fold. MSE/MAE/R² answer different questions. Below R² uses validation mean (sklearn convention); skill vs a training-mean baseline is separately computed in 11. Classification accuracy can hide minority-class failure; log loss measures probability quality. Folds overlap in training, so fold-score standard deviations are descriptive, not independent confidence intervals.

In [ ]:
classes = np.sort(y.unique()) if TASK == 'classification' else None
def evaluate(estimator, frame, truth):
    pred = estimator.predict(frame)
    if TASK == 'regression':
        return {'mse': mean_squared_error(truth, pred), 'mae': mean_absolute_error(truth, pred), 'r2': r2_score(truth, pred)}, pred, None
    probabilities = estimator.predict_proba(frame)
    fitted_classes = estimator.named_steps['model'].classes_
    assert set(fitted_classes) == set(classes), 'Training fold lacks classes'
    probabilities = probabilities[:, [list(fitted_classes).index(c) for c in classes]]
    metrics = {'logloss': log_loss(truth, probabilities, labels=classes), 'accuracy': accuracy_score(truth,pred), 'balanced_accuracy': balanced_accuracy_score(truth,pred)}
    if len(classes)==2 and pd.Series(truth).nunique()==2:
        j = list(classes).index(POS_LABEL)
        metrics['auc'] = roc_auc_score(np.asarray(truth)==POS_LABEL, probabilities[:,j])
        metrics['average_precision'] = average_precision_score(np.asarray(truth)==POS_LABEL, probabilities[:,j])
    return metrics, pred, probabilities
rows, oof, oof_prob, fold_models = [], {}, {}, {}
for name, candidate in models.items():
    predictions = pd.Series(index=X.index, dtype=float if TASK=='regression' else object)
    probs = np.full((len(X), len(classes)), np.nan) if classes is not None else None
    fold_models[name] = []
    for k, (a,b) in enumerate(cv):
        fitted = clone(candidate).fit(X.iloc[a], y.iloc[a])
        metrics, pred, probability = evaluate(fitted, X.iloc[b], y.iloc[b])
        rows.append(dict(model=name, fold=k, **metrics))
        predictions.iloc[b] = pred
        if probs is not None: probs[b] = probability
        fold_models[name].append(fitted)
    oof[name], oof_prob[name] = predictions, probs
fold_scores = pd.DataFrame(rows)
summary = fold_scores.groupby('model')[PRIMARY].agg(['mean','std']).sort_values('mean')
display(summary, fold_scores)
selected = summary.index[0]   # reconsider near ties for simplicity/stability
print('Provisional selection:', selected)

## 11 — Development diagnostics

Use out-of-fold residuals, not training residuals. Patterns vs fitted values suggest misspecification; a fan shape suggests changing error variance. Tail errors motivate robust loss or distributional predictions. Segment errors by time/entity/category and show sample counts. Negative R² is possible. For classification inspect confusion matrix and reliability diagram; tune thresholds on development data only. For temporal ACF sort residuals and aggregate simultaneous observations; ordinary row-order ACF on a panel is misleading.

In [ ]:
mask = oof[selected].notna()
diagnostics = dev.loc[mask].copy()
if TASK == 'regression':
    actual = y[mask].astype(float); pred = oof[selected][mask].astype(float)
    residual = actual - pred
    diagnostics['error'] = residual
    diagnostics['abs_error'] = residual.abs()
    baseline_error = ((actual-oof['baseline'][mask].astype(float))**2).sum()
    print('OOF skill vs fold-training-mean baseline:', 1-(residual**2).sum()/baseline_error if baseline_error else np.nan)
    fig, ax = plt.subplots(1,3,figsize=(14,4))
    ax[0].scatter(actual,pred,s=12,alpha=.5); ax[0].set(xlabel='Actual',ylabel='Predicted',title='OOF predictions')
    ax[1].scatter(pred,residual,s=12,alpha=.5); ax[1].axhline(0,color='black'); ax[1].set(xlabel='Predicted',ylabel='Residual',title='OOF residual pattern')
    ax[2].hist(residual,bins=30); ax[2].set(title='OOF residual tails',xlabel='Residual')
    plt.tight_layout(); plt.show()
    for c in cat[:2]:
        display(diagnostics.groupby(c,dropna=False).abs_error.agg(['count','mean']))
else:
    actual = y[mask]
    pred = oof[selected][mask].astype(y.dtype)
    ConfusionMatrixDisplay(confusion_matrix(actual,pred,labels=classes),display_labels=classes).plot(); plt.show()
    if len(classes)==2:
        from sklearn.calibration import calibration_curve
        p = oof_prob[selected][mask, list(classes).index(POS_LABEL)]
        observed, predicted = calibration_curve(actual==POS_LABEL,p,n_bins=8,strategy='quantile')
        plt.plot(predicted,observed,'o-',label='OOF'); plt.plot([0,1],[0,1],'--',label='ideal')
        plt.xlabel('Predicted event probability'); plt.ylabel('Observed frequency'); plt.legend(); plt.show()

## 12 — Optional small tuning / PCA / robust loss

Enable only a few hypotheses supported by EDA. This cell tunes on the existing development folds and changes the candidate model; rerun 10–11 to regenerate honest fold diagnostics (tuned-model OOF evaluation should use nested CV for an unbiased selection estimate). The untouched holdout remains the final assessment. Robust alternatives: HuberRegressor for numeric dense regression, or GradientBoostingRegressor(loss='huber'); positive-target log transformations via TransformedTargetRegressor require careful back-transformation since exp(E[log Y]) is not E[Y]. PCA goes inside preprocessing after scaling; never fit it globally.

In [ ]:
RUN_TUNING = False
if RUN_TUNING:
    name = 'ridge' if TASK=='regression' else 'logistic'
    grid = {'model__alpha':[.1,1,10,100]} if TASK=='regression' else {'model__C':[.1,1,10]}
    search = GridSearchCV(models[name],grid,scoring=SCORING,cv=cv,n_jobs=1)
    search.fit(X,y)
    display(pd.DataFrame(search.cv_results_)[['params','mean_test_score','std_test_score']])
    print(search.best_params_)
    # Adopt only after accounting for search/selection optimism.

## 13 — Importance and ablation

Why does a feature help? Permute raw columns on validation rows through the whole pipeline; this naturally groups a categorical feature's one-hot columns. Correlated features may substitute for each other and reduce measured importance; permutation can also create unrealistic combinations. Use drop-feature/group ablation across folds for incremental value. Neither coefficients nor importance establish causality. Partial dependence assumes meaningful feature combinations; correlated features make it misleading.

In [ ]:
a,b = cv[-1]
last_model = fold_models[selected][-1]
importance = permutation_importance(last_model,X.iloc[b],y.iloc[b],scoring=SCORING,n_repeats=5,random_state=SEED,n_jobs=1)
importance_table = pd.DataFrame({'feature':FEATURES,'importance':importance.importances_mean,'repeat_std':importance.importances_std}).sort_values('importance',ascending=False)
display(importance_table)
importance_table.set_index('feature').importance.sort_values().plot.barh(title='Last-fold permutation importance (score decrease)')
plt.xlabel('Score decrease'); plt.tight_layout(); plt.show()
RUN_ABLATION = False
if RUN_ABLATION:
    drop_feature = importance_table.iloc[0]['feature']
    reduced = clone(models[selected])
    reduced.set_params(preprocess=ColumnTransformer([
        (name, clone(transformer), [c for c in columns if c != drop_feature])
        for name, transformer, columns in preprocess.transformers
    ]))
    ablation_rows = []
    for k,(a,b) in enumerate(cv):
        fitted = clone(reduced).fit(X.iloc[a], y.iloc[a])
        metrics,_,_ = evaluate(fitted, X.iloc[b], y.iloc[b])
        ablation_rows.append(dict(fold=k, **metrics))
    display(pd.DataFrame(ablation_rows))
# Repeat across folds for stability. Optional PDP:
# PartialDependenceDisplay.from_estimator(last_model, X.iloc[b], features=[num[0]])

## 14 — Optional statistical tests

Tests need a stated null, sampling assumptions and effect size. Do not indiscriminately run every test. Welch t-test: difference in independent-group means, unequal variance allowed. Spearman: monotonic association, iid inference. Chi-square: categorical independence with adequate expected counts. ADF: unit-root null; KPSS: stationarity null; use chronological single series with explicit deterministic trend choice. Ljung–Box: residual serial dependence; Breusch–Pagan: heteroskedasticity in a specified regression. HAC standard errors address some serial/variance dependence for coefficient inference, not leakage or wrong functional form. Adjust p-values across related exploratory hypotheses; low p is not economic usefulness or causality. Block/cluster inference is required for dependent observations.

In [ ]:
RUN_TESTS = False
if RUN_TESTS:
    assert TASK == 'regression' and num
    c = num[0]
    valid = X[c].notna() & y.notna()
    association = stats.spearmanr(X.loc[valid,c],y[valid])
    print('Spearman statistic / iid p:', association)
    if cat and X[cat[0]].nunique()==2:
        levels = X[cat[0]].dropna().unique()
        print('Welch iid test:', stats.ttest_ind(y[X[cat[0]]==levels[0]],y[X[cat[0]]==levels[1]],equal_var=False))
    import statsmodels.api as sm
    from statsmodels.stats.diagnostic import het_breuschpagan, acorr_ljungbox
    from statsmodels.tsa.stattools import adfuller, kpss
    from statsmodels.stats.multitest import multipletests
    # Inference example: explicit numeric complete-case specification.
    z = pd.concat([X[num],y.rename('__y')],axis=1).dropna()
    design = sm.add_constant(z[num],has_constant='add')
    fit = sm.OLS(z['__y'],design).fit(cov_type='HC3')
    display(pd.DataFrame({'coef':fit.params,'robust_se':fit.bse,'p':fit.pvalues}))
    print('Breusch-Pagan LM / p / F / p:',het_breuschpagan(fit.resid,design))
    if SPLIT=='time':
        series = dev.groupby(TIME)[TARGET].mean().sort_index().dropna()
        print('ADF statistic / p:',adfuller(series,regression='c')[:2])
        print('KPSS statistic / p:',kpss(series,regression='c',nlags='auto')[:2])
        e = diagnostics.groupby(TIME).error.mean().sort_index().dropna()
        display(acorr_ljungbox(e,lags=[5,10],return_df=True))
    # Multiple comparisons: adjusted = multipletests(p_values, method='fdr_bh')[1]

## 15 — Optional uncertainty and robustness

Compare losses on the same observations (paired differences). Resample iid rows only for iid data; contiguous block resampling is a rough stationary time-series recipe, with block length checked for sensitivity. For panels resample time blocks including all entities or use appropriate cluster inference. This interval conditions on already fitted predictions: it does not include training/model-selection uncertainty. For correlated groups, bootstrap whole groups instead. Check different windows, simpler models, ablations, extreme-value sensitivity and performance by segment; repeated experimentation consumes validation information.

In [ ]:
RUN_UNCERTAINTY = False
if RUN_UNCERTAINTY:
    assert TASK=='regression'
    m = oof[selected].notna() & oof['baseline'].notna()
    delta = (y[m]-oof['baseline'][m].astype(float))**2 - (y[m]-oof[selected][m].astype(float))**2
    if SPLIT=='time':
        delta = pd.DataFrame({'t':dev.loc[m,TIME],'delta':delta}).groupby('t').delta.mean().sort_index()
    assert SPLIT!='group', 'Implement whole-group bootstrap for group dependence'
    values = np.asarray(delta)
    length = min(10,len(values)) if SPLIT=='time' else 1
    random = np.random.RandomState(SEED)
    samples = []
    for _ in range(500):
        starts = random.randint(0,len(values)-length+1,size=int(np.ceil(len(values)/length)))
        indices = np.concatenate([np.arange(s,s+length) for s in starts])[:len(values)]
        samples.append(values[indices].mean())
    print('Mean baseline loss reduction:',values.mean())
    print('Conditional bootstrap 95% interval:',np.quantile(samples,[.025,.975]))

## 16 — Optional shift and finance diagnostics

Before final evaluation use development fold-to-fold comparisons for shift. After freezing, descriptive development/holdout differences can explain failure, but adapting then requires a new holdout. KS tests compare marginal numeric distributions under iid assumptions; dependence, large n and many features complicate p-values. Covariate shift changes P(X), concept drift changes P(Y|X). Marginal drift alone does not prove a failed model.

For finance ask: horizon, execution delay, tradable prices, spread/fees/slippage, turnover, exposure, capacity and overlapping labels. Correlation/IC is not P&L. Sharpe must use equally spaced portfolio returns, not arbitrary prediction rows. Annual sqrt scaling assumes suitable dependence; show tail loss/drawdown and net costs. Do not claim profitability from anonymized regression data.

In [ ]:
RUN_FINANCE = False
if RUN_FINANCE:
    assert TASK=='regression'
    print('OOF Pearson IC:',stats.pearsonr(actual,pred)[0])
    print('OOF rank IC:',stats.spearmanr(actual,pred)[0])
    # Supply a REAL aligned net return series from an execution-aware backtest:
    # returns = positions.shift(1)*asset_returns - turnover*cost_per_unit
    # sharpe = returns.mean()/returns.std(ddof=1)*np.sqrt(periods_per_year)
    # equity = (1+returns).cumprod(); drawdown = equity/equity.cummax()-1
    # If many assets share a timestamp, construct portfolio returns first.

## 17 — Freeze and evaluate the holdout once

Enable RUN_HOLDOUT only once choices are frozen. Fit preprocessing/model on all development data; evaluate the reserved holdout once, alongside a training-only baseline. Avoid interpreting sklearn R² as skill against a training mean; both are reported. New feature choices based on this result invalidate its status as an untouched estimate. Inspect shift after this point without claiming a second tuned score is final.

In [ ]:
RUN_HOLDOUT = False
final_model = clone(models[selected]).fit(X,y)
if RUN_HOLDOUT:
    Xh = holdout[FEATURES].copy()
    for c in cat:
        Xh[c] = Xh[c].astype(object).where(Xh[c].notna(),np.nan)
    metrics, holdout_pred, holdout_prob = evaluate(final_model,Xh,holdout[TARGET])
    base = clone(models['baseline']).fit(X,y)
    baseline_metrics, baseline_pred, _ = evaluate(base,Xh,holdout[TARGET])
    display(pd.DataFrame([dict(model=selected,**metrics),dict(model='baseline',**baseline_metrics)]))
    if TASK=='regression':
        denom = np.sum((holdout[TARGET]-baseline_pred)**2)
        print('Holdout skill vs training baseline:',1-np.sum((holdout[TARGET]-holdout_pred)**2)/denom if denom else np.nan)
    shift = []
    for c in num:
        left,right = X[c].dropna(),Xh[c].dropna()
        if len(left) and len(right):
            ks = stats.ks_2samp(left,right)
            shift.append({'feature':c,'dev_mean':left.mean(),'holdout_mean':right.mean(),'ks_stat':ks.statistic,'iid_p':ks.pvalue})
    display(pd.DataFrame(shift))

## 18 — Prediction, export and research summary

Prediction uses the exact saved feature schema and fitted preprocessing. Preserve row IDs and ordering. Do not refit on new prediction data. Store artifacts only in an appropriate private location; never commit interview/company data or fitted sensitive artifacts to this public repository. Pickle/joblib files must come from trusted sources. Export code is off by default.

Present: objective and observation grain → 2–4 findings that changed decisions → split and leakage controls → baseline/model fold table → diagnostics/importance → uncertainty and limitations → next experiments. Explain why the selected model is justified, what could invalidate the finding and what you would test with another day. Keep an experiment log of hypothesis, change, fold result, decision and time. No signal is a valid outcome when supported by honest evaluation.

In [ ]:
RUN_EXPORT = False
if RUN_EXPORT:
    import joblib
    new = pd.read_csv('new_data.csv')
    assert set(FEATURES).issubset(new.columns)
    new_X = new[FEATURES].copy()
    for c in cat:
        new_X[c] = new_X[c].astype(object).where(new_X[c].notna(),np.nan)
    result = pd.DataFrame({'row_id':new.index,'prediction':final_model.predict(new_X)})
    if TASK=='classification':
        probabilities = final_model.predict_proba(new_X)
        for i,c in enumerate(final_model.named_steps['model'].classes_):
            result['probability_'+str(c)] = probabilities[:,i]
    result.to_csv('predictions.csv',index=False)
    joblib.dump({'model':final_model,'features':FEATURES,'task':TASK},'model.joblib')
display(summary)
print('Next: summarize evidence, limitations, and one next experiment.')